# Visualization Config Generator

Converts **normalized** CSV files from `normalized/` into visualization config skeleton
JSON files, writing output to `output/<form_id>.json`.

Run `normalize_csv.ipynb` first to produce the normalized CSVs.

**Normalized CSV schema**: `group`, `indicator`, `calculation`
(all rows in normalized are valid — no feasibility filter needed).

**Rules applied**:
- `question_name` references are parsed **directly from the `calculation` formula**.
  When authored in **VizCalc** ([`README.md`](README.md)), inputs are explicit
  `[question_name]` tokens read in order (zero false positives, nothing dropped). Rows
  still in legacy free prose fall back to scanning bare snake_case tokens against the
  form source index. There is no separate `used_questions` column — the formula is the
  single source of truth.
- The **chart type** comes from the VizCalc OUTPUT function (outermost `UPPERCASE(...)`):
  `COUNT/COUNTDISTINCT/RECENT` → card, `PERCENT` → metric_card, `DISTRIBUTION` →
  half_doughnut, `VALUE` → bar/dot_strip, `COMPLIANT` → stack_bar/compliance, `RANK` →
  ranking, `MAP` → map filter. Legacy-prose rows (no OUTPUT token) use the indicator-label
  heuristics in `build_item`, so their output is unchanged during the transition.
- **Most question references use `question_name`** (never `question_id` /
  `date_question_id`). `CAPACITY_COMPARE` is the exception because it compares
  registration and monitoring forms directly, so it emits `form_id` + `question_id`
  measure APIs. The backend `/values` endpoint serves question-name items from
  `mv_cross_form_latest`:
  - distribution / per-parent / average → `group_by` + `value_type`
  - card parent counts → `sum_by=parent_id`
  - option-value KPI counts → `option_value` (+ `sum_by`)
  - recency cards → `rolling_months` / `from_date` / `to_date`
- `form_id` is kept **only** for form-level references that are not question
  parameters: registration-count cards, table `source` form, map `source_form_id`.

**Run from** `scripts/visualization-config/` (adjust `REPO_ROOT` if needed).

In [ ]:
import json
import re
import csv
from pathlib import Path
from collections import defaultdict

REPO_ROOT = Path("../..").resolve()
FORMS_DIR = REPO_ROOT / "backend/source/forms"
SOURCE_DIR = Path("normalized")   # normalized CSVs with question_name strings
OUTPUT_DIR = Path("../../frontend/src/config/visualizations")

OUTPUT_DIR.mkdir(exist_ok=True)

assert FORMS_DIR.exists(), f"Forms dir not found: {FORMS_DIR}"
assert SOURCE_DIR.exists(), f"Normalized dir not found: {SOURCE_DIR} — run normalize_csv.ipynb first"

print(f"REPO_ROOT : {REPO_ROOT}")
print(f"FORMS_DIR : {FORMS_DIR}")
print(f"SOURCE_DIR: {SOURCE_DIR}  (normalized)")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

In [ ]:
# Build lookup maps from the PROD form source JSONs
#
# Only `*.prod.json` family files are read (pattern <prefix>_<formId>[.monitoring].prod.json);
# example-*/test fixtures are ignored. This keeps the index to the deployed forms.
#
# fid_map   : form_id  -> form_name
# qname_map : question_name -> {question_id, question_type, form_id}
# forms_raw : form_id  -> full parsed form dict (used for family resolution below)
#
# When the same question_name appears in multiple forms (e.g., turbidity_ntu in
# both Monitoring and Quick Monitoring), prefer the comprehensive monitoring form.

fid_map: dict = {}    # int -> str
qname_map: dict = {}  # str -> {question_id, question_type, form_id}
forms_raw: dict = {}  # int -> full form dict


def _index_form(form_data: dict) -> None:
    form_id = form_data["id"]
    form_name = form_data.get("form") or form_data.get("name", str(form_id))
    fid_map[form_id] = form_name
    forms_raw[form_id] = form_data
    for qg in form_data.get("question_groups", []):
        for q in qg.get("questions", []):
            qid = q["id"]
            name = q.get("name", "")
            qtype = q.get("type", "")
            if not name:
                continue
            candidate = {"question_id": qid, "question_type": qtype, "form_id": form_id}
            existing = qname_map.get(name)
            if existing is None:
                qname_map[name] = candidate
            else:
                # Prefer comprehensive monitoring form over quick monitoring form
                existing_form = fid_map.get(existing["form_id"], "")
                if "Quick" in existing_form and "Quick" not in form_name and "Monitoring" in form_name:
                    qname_map[name] = candidate


# Read only deployed family forms: <prefix>_<formId>[.monitoring].prod.json
for f in sorted(FORMS_DIR.glob("*.prod.json")):
    try:
        _index_form(json.loads(f.read_text()))
    except Exception as e:
        print(f"Warning: {f.name}: {e}")

print(f"Indexed {len(qname_map)} unique question names across {len(fid_map)} forms.")
print("\nForms:")
for fid, fname in sorted(fid_map.items()):
    parent = forms_raw[fid].get("parent_id")
    suffix = f"  (parent: {parent})" if parent else "  (registration)"
    print(f"  {fid}  {fname}{suffix}")

In [ ]:
# CSV utilities

# VizCalc input token: a question is always written [question_name] (see README.md).
INPUT_RE = re.compile(r'\[([a-z][a-z0-9_]*)\]')
# Legacy free-prose fallback: bare snake_case tokens, matched against qname_map.
QNAME_RE = re.compile(r'\b[a-z][a-z0-9_]*\b')


def load_rows(csv_path: Path) -> list:
    """Load all rows from a normalized CSV (group, indicator, calculation).

    All rows in the normalized CSV are valid — no feasibility filter needed.
    """
    with open(csv_path) as f:
        return list(csv.DictReader(f))


def row_qinfos(row: dict) -> list:
    """Extract question infos from the row's 'calculation' formula.

    VizCalc inputs are explicitly delimited as [question_name], so they are read
    exactly and in order — no false positives from prose, nothing silently dropped.
    For rows still in legacy free prose (no brackets), fall back to scanning bare
    snake_case tokens against qname_map.

    qinfos[0] is the formula's primary subject question.
    Returns a deduplicated list of {question_id, question_name, question_type, form_id}.
    """
    text = row.get("calculation", "")
    tokens = INPUT_RE.findall(text)
    if not tokens:
        tokens = QNAME_RE.findall(text)
    seen, result = set(), []
    for token in tokens:
        if token in seen or token not in qname_map:
            continue
        seen.add(token)
        info = qname_map[token]
        result.append({
            "question_id":   info["question_id"],
            "question_name": token,
            "question_type": info["question_type"],
            "form_id":       info["form_id"],
        })
    return result


def _slug(text: str) -> str:
    """Convert text to a safe lowercase identifier (max 40 chars)."""
    s = re.sub(r'[^a-zA-Z0-9\s_]', '', text.lower())
    s = re.sub(r'\s+', '_', s.strip())
    return s[:40].strip('_')


# Sanity check: VizCalc (bracketed) vs legacy prose
vizcalc = 'COUNTDISTINCT(COALESCE([plant_name],[geolocation]))'
legacy = "COUNT DISTINCT Registration records by plant_name (fall back to geolocation)."
for label, sample in [("VizCalc", vizcalc), ("legacy", legacy)]:
    names = [i["question_name"] for i in row_qinfos({"calculation": sample})]
    print(f"{label:8} {sample!r}\n  -> {names}")

In [ ]:
# Water quality parameter metadata
# keyword (searched in lowercased indicator) -> (display_label, threshold, unit)

WQ_PARAMS = {
    "e_coli":            ("E.coli",            {"max": 0},                "CFU/100mL"),
    "e.coli":            ("E.coli",            {"max": 0},                "CFU/100mL"),
    "e-coli":            ("E.coli",            {"max": 0},                "CFU/100mL"),
    "total_coliform":    ("Total coliform",    {"max": 0},                "CFU/100mL"),
    "total coliform":    ("Total coliform",    {"max": 0},                "CFU/100mL"),
    "fecal_coliform":    ("Fecal coliform",    {"max": 0},                "CFU/100mL"),
    "fecal coliform":    ("Fecal coliform",    {"max": 0},                "CFU/100mL"),
    "turbidity":         ("Turbidity",         {"max": 5},                "NTU"),
    "residual_chlorine": ("Residual chlorine", {"min": 0.2, "max": 0.5}, "mg/L"),
    "residual chlorine": ("Residual chlorine", {"min": 0.2, "max": 0.5}, "mg/L"),
    "conductivity":      ("Conductivity",      {"max": 1000},             "\u00b5S/cm"),
    "salinity":          ("Salinity",          {"max": 1},                "ppt"),
    "temperature":       ("Temperature",       {"max": 30},               "\u00b0C"),
    "bod":               ("BOD",               {"max": 40},               "mg/L"),
    "cod":               ("COD",               {"max": 100},              "mg/L"),
    "tds":               ("TDS",               {"max": 1000},             "mg/L"),
}


def match_wq_param(indicator: str):
    """Return (label, threshold, unit) if indicator is a WQ parameter, else None."""
    ind = indicator.lower()
    if re.search(r'\bph\b', ind):
        return ("pH", {"min": 6.5, "max": 8.5}, "pH")
    for key, val in WQ_PARAMS.items():
        if key in ind:
            return val
    return None


def _threshold_desc(threshold: dict, unit: str) -> str:
    if "min" in threshold and "max" in threshold:
        return f"Acceptable range {threshold['min']}\u2013{threshold['max']} {unit}"
    if "max" in threshold:
        v = threshold['max']
        return f"Acceptance threshold \u2264 {v} {unit}" if v == 0 else f"Acceptance threshold < {v} {unit}"
    return ""

In [ ]:
# Skeleton chart item builder

def build_item(row: dict, order: int, col_span: int = 12, entity_label: str = "site") -> dict:
    """Build a skeleton config item from one normalized CSV row.

    Question references always use question_name (never question_id). The
    backend /values endpoint serves every shape below from mv_cross_form_latest
    via question_name:
    - distribution / per-parent / average  -> group_by + value_type
    - card parent counts                   -> sum_by=parent_id
    - option-value counts (KPIs)           -> option_value + sum_by
    - recency cards                        -> rolling_months / from_date

    Only form-level references (registration counts, table/map source form)
    keep form_id, since those are not question parameters.
    """
    indicator = row["indicator"].strip()
    ind_lower = indicator.lower()
    qinfos = row_qinfos(row)
    primary = qinfos[0] if qinfos else {}
    p_qtype = primary.get("question_type", "")
    p_qname = primary.get("question_name", "")
    p_fid = primary.get("form_id")
    item_id = _slug(indicator)

    # The date question among the row's questions (for recency cards).
    date_qinfo = next(
        (q for q in qinfos if q.get("question_type") == "date"), primary
    )
    date_qname = date_qinfo.get("question_name", "")

    # Total count card (registration count — no question, keep form_id)
    if re.match(r'^total\s', ind_lower):
        return {
            "id": f"kpi_{item_id}",
            "chart_type": "card",
            "order": order,
            "col_span": col_span,
            "label": indicator,
            "api": {
                "form_id": p_fid or "__TODO__",
                "monitoring": "latest",
            },
        }

    # Rolling 12-month card -> count parents monitored recently by date question
    if "(12 mo)" in ind_lower or "12 mo" in ind_lower:
        return {
            "id": f"kpi_{item_id}",
            "chart_type": "card",
            "order": order,
            "col_span": col_span,
            "label": indicator,
            "api": {
                "question_name": date_qname or p_qname or "__TODO_date_question_name__",
                "sum_by": "parent_id",
                "rolling_months": 12,
            },
        }

    # Compliance by parameter (stack_bar)
    if "compliance by parameter" in ind_lower or (
        "compliance" in ind_lower and "parameter" in ind_lower
    ):
        return {
            "id": f"chart_{item_id}",
            "chart_type": "stack_bar",
            "order": order,
            "col_span": 24,
            "config": {"title": indicator},
            "compute": "compliance",
            "include_unanswered": True,
            "params_ref": ["__TODO_list_param_ids__"],
            "globals_ref": "__TODO_wq_globals_ref__",
        }

    # Compliance KPI card
    if "compliance" in ind_lower:
        return {
            "id": f"kpi_{item_id}",
            "chart_type": "card",
            "order": order,
            "col_span": col_span,
            "label": indicator,
            "color": "#64A73B",
            "compute": "compliance_kpi",
            "params_ref": ["__TODO_list_param_ids__"],
            "globals_ref": "__TODO_wq_globals_ref__",
            "denominator_api": {"form_id": "__TODO_reg_form_id__"},
        }

    # Water quality numeric parameter -> dot_strip
    # Uses question_name: mv_cross_form_latest gives latest value per parent — no repeat_agg needed.
    wq = match_wq_param(indicator)
    if wq and p_qtype == "number" and p_qname:
        wq_label, threshold, unit = wq
        return {
            "id": f"param_{item_id}",
            "chart_type": "dot_strip",
            "order": order,
            "col_span": col_span,
            "label": wq_label,
            "description": _threshold_desc(threshold, unit),
            "config": {
                "title": wq_label,
                "xAxisLabel": unit,
                "entity_label": entity_label,
            },
            "threshold": threshold,
            "api": {
                "question_name": p_qname,
                "group_by": "parent_id",
                "monitoring": "latest",
            },
        }

    # % metric card (operationality option distribution + target)
    if ind_lower.startswith("%") or re.match(r'^%\s', ind_lower):
        return {
            "id": f"kpi_{item_id}",
            "chart_type": "metric_card",
            "order": order,
            "col_span": col_span,
            "label": indicator,
            "api": {
                "question_name": p_qname or "__TODO_question_name__",
                "group_by": "option",
                "include_unanswered": True,
            },
            "show_percentage": True,
            "target_group": "__TODO_option_value__",
            "_note": "Verify: may require multi-question operationality compute logic",
        }

    # No sample / No water-quality check card -> count parents whose latest == no
    if re.search(r'no\s+(water.quality|sample|check)', ind_lower):
        return {
            "id": f"kpi_{item_id}",
            "chart_type": "card",
            "order": order,
            "col_span": col_span,
            "label": indicator,
            "api": {
                "question_name": p_qname or "__TODO_question_name__",
                "option_value": "no",
                "sum_by": "parent_id",
                "monitoring": "latest",
            },
        }

    # Critical issues card -> count parents whose latest == yes
    if "critical" in ind_lower:
        return {
            "id": f"kpi_{item_id}",
            "chart_type": "card",
            "order": order,
            "col_span": col_span,
            "label": indicator,
            "color": "#e41a1c",
            "api": {
                "question_name": p_qname or "__TODO_question_name__",
                "option_value": "yes",
                "sum_by": "parent_id",
                "monitoring": "latest",
            },
        }

    # Option / multiple_option distribution -> question_name
    if p_qtype in ("option", "multiple_option") and p_qname:
        use_combo = p_qtype == "multiple_option"
        return {
            "id": f"chart_{item_id}",
            "chart_type": "half_doughnut",
            "order": order,
            "col_span": col_span,
            "config": {"title": indicator},
            "api": {
                "question_name": p_qname,
                "group_by": "option_combo" if use_combo else "option",
                "monitoring": "latest",
                "include_unanswered": True,
            },
        }

    # Number single-question -> question_name (bar, scatter, etc.)
    if p_qtype == "number" and p_qname:
        return {
            "id": f"chart_{item_id}",
            "chart_type": "bar",
            "order": order,
            "col_span": col_span,
            "config": {
                "title": indicator,
                "xAxisLabel": "__TODO_unit__",
                "yAxisLabel": "count",
            },
            "api": {
                "question_name": p_qname,
                "group_by": "parent_id",
                "monitoring": "latest",
            },
        }

    # Generic fallback
    qnames = [q['question_name'] for q in qinfos]
    return {
        "id": f"chart_{item_id}",
        "chart_type": "__TODO__",
        "order": order,
        "col_span": col_span,
        "label": indicator,
        "_note": f"No automatic mapping. question_names: {qnames}",
        "api": (
            {"question_name": p_qname}
            if p_qname
            else {"_todo": "No questions found in calculation formula"}
        ),
    }

In [ ]:
# VizCalc output-function dispatch
#
# When a row's `calculation` is authored in VizCalc (README.md), the OUTPUT is the
# outermost UPPERCASE(...) function and it determines the chart type directly —
# no string-heuristics on the indicator label. A row is treated as VizCalc only when
# it BOTH starts with a known OUTPUT function AND contains at least one [question_name]
# input; legacy free prose (no bracketed input — even something like "COUNT(Yes) of …")
# falls through to the heuristic build_item below (output stays byte-identical during
# the transition).

VIZCALC_OUTPUTS = {
    "COUNT", "COUNTDISTINCT", "RECENT", "PERCENT",
    "DISTRIBUTION", "OPTION_COUNTS", "PROCESS_COUNTS", "CAPACITY_COMPARE", "DATE_HISTOGRAM", "VALUE_BUCKETS", "STAGE_FLOW", "VALUE", "COMPLIANT", "RANK", "MAP",
}

_OUTER_FN_RE = re.compile(r'\s*([A-Z][A-Z0-9_]*)\(')


def vizcalc_outer(calc: str):
    """Parse the outermost UPPERCASE(...) output function of a VizCalc formula.

    Returns (fn_name, [top_level_arg_strings]). Returns (None, []) when the formula
    is not VizCalc (legacy prose / no leading UPPERCASE-function), so callers can
    fall back to the heuristic builder.
    """
    if not calc:
        return (None, [])
    m = _OUTER_FN_RE.match(calc)
    if not m:
        return (None, [])
    fn = m.group(1)
    depth, buf, args = 1, [], []   # depth 1 = inside the outer (...)
    for ch in calc[m.end():]:
        if depth <= 0:
            break
        if ch in "([":
            depth += 1
            buf.append(ch)
        elif ch in ")]":
            depth -= 1
            if depth > 0:
                buf.append(ch)
        elif ch == "," and depth == 1:
            args.append("".join(buf).strip())
            buf = []
        else:
            buf.append(ch)
    tail = "".join(buf).strip()
    if tail:
        args.append(tail)
    return (fn, args)


def _first_number(args: list):
    """First numeric literal among args (int when whole, else float); None if none."""
    for a in args:
        m = re.search(r'-?\d+(?:\.\d+)?', a)
        if m:
            val = float(m.group())
            return int(val) if val.is_integer() else val
    return None


def _option_literal(calc: str):
    """First double-quoted option/text literal compared with '=' , else None."""
    m = re.search(r'=\s*"([^"]+)"', calc)
    return m.group(1) if m else None


def _process_count_segment(arg: str, index: int):
    """Parse PROCESS_COUNTS arg: [question]="value" AS "Label"."""
    qmatch = INPUT_RE.search(arg)
    qname = qmatch.group(1) if qmatch else "__TODO_question_name__"
    label_match = re.search(r'\s+AS\s+"([^"]+)"\s*$', arg, flags=re.IGNORECASE)
    label = label_match.group(1) if label_match else qname.replace("_", " ").title()
    key = _slug(label) or f"segment_{index + 1}"
    api = {"question_name": qname, "sum_by": "parent_id", "monitoring": "latest"}
    opt = _option_literal(arg)
    if opt:
        api["option_value"] = opt
    return {"key": key, "label": label, "api": api}


def _stage_flow_segment(arg: str, index: int):
    """Parse STAGE_FLOW arg: [question] AS "Label"."""
    qmatch = INPUT_RE.search(arg)
    qname = qmatch.group(1) if qmatch else "__TODO_question_name__"
    label_match = re.search(r'\s+AS\s+"([^"]+)"\s*$', arg, flags=re.IGNORECASE)
    label = label_match.group(1) if label_match else qname.replace("_", " ").title()
    key = _slug(label) or f"stage_{index + 1}"
    return {
        "key": key,
        "label": label,
        "api": {"question_name": qname, "group_by": "parent_id", "monitoring": "latest"},
    }


def _value_bucket(arg: str):
    """Parse VALUE_BUCKETS bucket arg: 0, 1, 2, or 4+."""
    label = arg.strip().strip('"')
    plus = re.match(r'^(-?\d+(?:\.\d+)?)\+$', label)
    if plus:
        raw = float(plus.group(1))
        value = int(raw) if raw.is_integer() else raw
        return {"label": label, "min": value}
    raw = float(label)
    value = int(raw) if raw.is_integer() else raw
    return {"label": label, "value": value}


def build_vizcalc_item(row: dict, output: str, args: list, qinfos: list,
                       order: int, col_span: int, entity_label: str):
    """Build a chart item from a VizCalc row by its OUTPUT function.

    Question references always use question_name. Returns None if the output has
    no item-level mapping here (e.g. MAP, handled in build_config), so the caller
    falls back to the legacy heuristic builder.
    """
    indicator = row["indicator"].strip()
    item_id = _slug(indicator)
    calc = row.get("calculation", "")
    primary = qinfos[0] if qinfos else {}
    p_qname = primary.get("question_name", "")
    p_qtype = primary.get("question_type", "")
    p_fid = primary.get("form_id")
    date_qinfo = next((q for q in qinfos if q.get("question_type") == "date"), primary)
    date_qname = date_qinfo.get("question_name", "")

    if output == "COUNTDISTINCT":
        # distinct parent / registration count — form-level, keep form_id
        return {
            "id": f"kpi_{item_id}", "chart_type": "card",
            "order": order, "col_span": col_span, "label": indicator,
            "api": {"form_id": p_fid or "__TODO__", "monitoring": "latest"},
        }

    if output == "RECENT":
        months = _first_number(args) or 12
        return {
            "id": f"kpi_{item_id}", "chart_type": "card",
            "order": order, "col_span": col_span, "label": indicator,
            "api": {
                "question_name": date_qname or p_qname or "__TODO_date_question_name__",
                "sum_by": "parent_id",
                "rolling_months": months,
            },
        }

    if output == "COUNT":
        api = {
            "question_name": p_qname or "__TODO_question_name__",
            "sum_by": "parent_id",
            "monitoring": "latest",
        }
        opt = _option_literal(calc)
        if opt:
            api["option_value"] = opt
        return {
            "id": f"kpi_{item_id}", "chart_type": "card",
            "order": order, "col_span": col_span, "label": indicator,
            "api": api,
        }

    if output == "PERCENT":
        return {
            "id": f"kpi_{item_id}", "chart_type": "metric_card",
            "order": order, "col_span": col_span, "label": indicator,
            "api": {
                "question_name": p_qname or "__TODO_question_name__",
                "group_by": "option",
                "include_unanswered": True,
            },
            "show_percentage": True,
            "target_group": _option_literal(calc) or "__TODO_option_value__",
        }

    if output == "DISTRIBUTION":
        flat = len(args) > 1 and args[1].strip().upper() == "FLAT"
        use_combo = p_qtype == "multiple_option" and not flat
        return {
            "id": f"chart_{item_id}", "chart_type": "half_doughnut",
            "order": order, "col_span": col_span,
            "config": {"title": indicator},
            "api": {
                "question_name": p_qname or "__TODO_question_name__",
                "group_by": "option_combo" if use_combo else "option",
                "monitoring": "latest",
                "include_unanswered": True,
            },
        }

    if output == "OPTION_COUNTS":
        return {
            "id": f"chart_{item_id}", "chart_type": "bar",
            "order": order, "col_span": col_span,
            "orientation": "horizontal",
            "config": {
                "title": indicator,
                "xAxisLabel": f"# of {entity_label}s",
                "yAxisLabel": "option",
                "grid": {"left": 160, "right": 32, "top": 32, "bottom": 48, "containLabel": False},
            },
            "api": {
                "question_name": p_qname or "__TODO_question_name__",
                "group_by": "option",
                "monitoring": "latest",
            },
        }

    if output == "PROCESS_COUNTS":
        return {
            "id": f"chart_{item_id}", "chart_type": "bar",
            "order": order, "col_span": col_span,
            "compute": "process_counts",
            "orientation": "horizontal",
            "sort": "desc",
            "config": {"title": indicator, "xAxisLabel": entity_label + "s", "yAxisLabel": "process"},
            "segments": [_process_count_segment(arg, i) for i, arg in enumerate(args)],
        }

    if output == "CAPACITY_COMPARE":
        prod_match = INPUT_RE.search(args[0]) if len(args) > 0 else None
        design_match = INPUT_RE.search(args[1]) if len(args) > 1 else None
        production_q = prod_match.group(1) if prod_match else "__TODO_production_question_name__"
        design_q = design_match.group(1) if design_match else "__TODO_design_question_name__"
        qinfo_by_name = {q.get("question_name"): q for q in qinfos}

        def _measure_api(qname, monitoring=False):
            info = qinfo_by_name.get(qname, {})
            api = {
                "form_id": info.get("form_id", "__TODO_form_id__"),
                "question_id": info.get("question_id", "__TODO_question_id__"),
                "repeat_agg": "sum",
            }
            if monitoring:
                api["monitoring"] = "latest"
            return api

        return {
            "id": f"chart_{item_id}", "chart_type": "bar",
            "order": order, "col_span": col_span,
            "compute": "capacity_compare",
            "config": {"title": indicator, "xAxisLabel": "Capacity", "yAxisLabel": "ML/day"},
            "measures": [
                {"key": "design", "label": "Design capacity", "api": _measure_api(design_q)},
                {"key": "production", "label": "Production", "api": _measure_api(production_q, monitoring=True)},
            ],
            "detail": {"mode": "filtered_scope", "enabled_by": "administration_filter"},
        }

    if output == "DATE_HISTOGRAM":
        qmatch = INPUT_RE.search(args[0]) if args else None
        qname = qmatch.group(1) if qmatch else p_qname or "__TODO_date_question_name__"
        months = _first_number(args[1:]) or 18
        return {
            "id": f"chart_{item_id}", "chart_type": "bar",
            "order": order, "col_span": col_span,
            "compute": "date_histogram",
            "description": "Green = inspected ≤ 6 months ago · Amber = 7–12 months · Red = > 12 months (overdue).",
            "config": {
                "title": indicator,
                "xAxisLabel": "Month",
                "yAxisLabel": "count",
                "xAxis": {"axisLabel": {"interval": 1, "rotate": 45}, "nameGap": 72},
            },
            "api": {"question_name": qname, "group_by": "parent_id", "monitoring": "latest"},
            "display": {
                "mode": "date_histogram",
                "months": months,
                "overdue_label": f"> {months} mo",
                "colors": {"recent": "#2fb36d", "watch": "#f5a623", "overdue": "#d93c35"},
            },
        }

    if output == "VALUE_BUCKETS":
        qmatch = INPUT_RE.search(args[0]) if args else None
        qname = qmatch.group(1) if qmatch else p_qname or "__TODO_number_question_name__"
        buckets = [_value_bucket(arg) for arg in args[1:]]
        return {
            "id": f"chart_{item_id}", "chart_type": "bar",
            "order": order, "col_span": col_span,
            "config": {"title": indicator, "xAxisLabel": "Value", "yAxisLabel": "count"},
            "api": {"question_name": qname, "group_by": "parent_id", "monitoring": "latest"},
            "display": {"mode": "value_buckets", "buckets": buckets},
        }

    if output == "STAGE_FLOW":
        return {
            "id": f"chart_{item_id}", "chart_type": "custom_component",
            "component": "StageFlowWidget",
            "order": order, "col_span": 24,
            "title": indicator,
            "height": 280,
            "root_label": f"All {entity_label}s",
            "summary_template": f"Of {{total}} {entity_label}s: {{final}} reached the final stage.",
            "denominator_api": {"form_id": "__TODO_reg_form_id__"},
            "stages": [_stage_flow_segment(arg, i) for i, arg in enumerate(args)],
        }

    if output == "VALUE":
        wq = match_wq_param(indicator)
        if wq and p_qtype == "number" and p_qname:
            wq_label, threshold, unit = wq
            return {
                "id": f"param_{item_id}", "chart_type": "dot_strip",
                "order": order, "col_span": col_span, "label": wq_label,
                "description": _threshold_desc(threshold, unit),
                "config": {"title": wq_label, "xAxisLabel": unit, "entity_label": entity_label},
                "threshold": threshold,
                "api": {"question_name": p_qname, "group_by": "parent_id", "monitoring": "latest"},
            }
        return {
            "id": f"chart_{item_id}", "chart_type": "bar",
            "order": order, "col_span": col_span,
            "config": {"title": indicator, "xAxisLabel": "__TODO_unit__", "yAxisLabel": "count"},
            "api": {
                "question_name": p_qname or "__TODO_question_name__",
                "group_by": "parent_id",
                "monitoring": "latest",
            },
        }

    if output == "COMPLIANT":
        if "parameter" in indicator.lower():
            return {
                "id": f"chart_{item_id}", "chart_type": "stack_bar",
                "order": order, "col_span": 24,
                "config": {"title": indicator},
                "compute": "compliance",
                "include_unanswered": True,
                "params_ref": ["__TODO_list_param_ids__"],
                "globals_ref": "__TODO_wq_globals_ref__",
            }
        return {
            "id": f"kpi_{item_id}", "chart_type": "card",
            "order": order, "col_span": col_span, "label": indicator,
            "color": "#64A73B",
            "compute": "compliance_kpi",
            "params_ref": ["__TODO_list_param_ids__"],
            "globals_ref": "__TODO_wq_globals_ref__",
            "denominator_api": {"form_id": "__TODO_reg_form_id__"},
        }

    if output == "RANK":
        n = _first_number(args) or 8
        direction = "desc" if any("DESC" in a.upper() for a in args) else "asc"
        return {
            "id": f"rank_{item_id}", "chart_type": "ranking",
            "order": order, "col_span": col_span, "label": indicator,
            "api": {
                "question_name": date_qname or p_qname or "__TODO_date_question_name__",
                "sort": direction,
                "limit": n,
                "monitoring": "latest",
            },
        }

    # MAP (handled as map filters in build_config) or unmapped output -> fall back
    return None


# Wrap the heuristic builder: VizCalc rows dispatch by OUTPUT, legacy prose falls back.
_build_item_legacy = build_item


def build_item(row: dict, order: int, col_span: int = 12, entity_label: str = "site") -> dict:
    """Dispatch to the VizCalc OUTPUT builder; fall back to legacy heuristics.

    A row authored in VizCalc has an outermost UPPERCASE(...) output function AND at
    least one [question_name] input — that pair picks the chart type deterministically.
    Legacy free-prose rows (no bracketed input, even if they happen to start with a
    word like "COUNT(Yes) of …") use the heuristic _build_item_legacy, so their output
    is unchanged.
    """
    calc = row.get("calculation", "")
    output, args = vizcalc_outer(calc)
    if output in VIZCALC_OUTPUTS and INPUT_RE.search(calc):
        item = build_vizcalc_item(
            row, output, args, row_qinfos(row), order, col_span, entity_label
        )
        if item is not None:
            return item
    return _build_item_legacy(row, order, col_span, entity_label)

In [ ]:
# Group -> tab/section role mapping

def group_role(group_name: str) -> str:
    """Return a canonical role string for a CSV group name."""
    g = group_name.lower()
    if "cards" in g:
        return "kpi_cards"
    if re.search(r'operational map|\bmap\b', g):
        return "map"
    if "at a glance" in g or "at-a-glance" in g:
        return "at_a_glance"
    if "monitoring status" in g:
        return "tab_monitoring"
    if "water quality" in g:
        return "tab_water_quality"
    if "treatment" in g or "functionality" in g:
        return "tab_treatment"
    if g.startswith("individual"):
        return "tab_individual"
    if re.match(r'^all\s', g):
        return "tab_all"
    return "section"


def groups_ordered(rows: list) -> list:
    """Return unique group names in order of first appearance."""
    seen, result = set(), []
    for r in rows:
        g = r["group"]
        if g not in seen:
            seen.add(g)
            result.append(g)
    return result

In [ ]:
# Full config skeleton builder

def _standby_facility_map_filter(primary: dict, label: str) -> dict:
    """Formula-backed map filter for numeric standby pump counts."""
    return {
        "type": "select",
        "key": "standby_facility",
        "label": label,
        "form_id": primary["form_id"],
        "formula": {
            "buckets": [
                {
                    "value": "standby_available",
                    "label": "Standby available",
                    "all_of": [{"question_name": "num_pumps_standby", "op": ">", "value": 0}],
                },
                {
                    "value": "no_standby",
                    "label": "No standby",
                    "all_of": [{"question_name": "num_pumps_standby", "op": "<=", "value": 0}],
                },
            ],
            "default": {"value": "_no_info", "label": "No information available"},
        },
        "color_map": {
            "standby_available": "#64A73B",
            "no_standby": "#e41a1c",
            "_no_info": "#999999",
        },
    }


def build_config(form_id: int, rows: list, form_name: str, entity_label: str = "site") -> dict:
    """Build a visualization config skeleton from normalized CSV rows."""

    by_group = defaultdict(list)
    for r in rows:
        by_group[r["group"]].append(r)

    groups = groups_ordered(rows)
    items = []
    order = 1

    # Standard filter bar
    items.append({
        "id": "filters_main",
        "chart_type": "filter_bar",
        "order": order,
        "col_span": 24,
        "items": [
            {
                "id": "filter_date",
                "chart_type": "filter_date",
                "order": 1,
                "label": "Monitoring Period",
                "date_question_names": {"monitoring": "__TODO_inspection_date_qname__"},
            },
            {
                "id": "filter_administration",
                "chart_type": "filter_administration",
                "order": 2,
                "label": "Location",
            },
        ],
    })
    order += 1

    # KPI cards (top-level, 4 per row = col_span 6)
    for g in groups:
        if group_role(g) == "kpi_cards":
            for r in by_group[g]:
                items.append(build_item(r, order, col_span=6, entity_label=entity_label))
                order += 1

    # Map
    for g in groups:
        if group_role(g) == "map":
            map_filters = []
            for r in by_group[g]:
                qinfos = row_qinfos(r)
                primary = qinfos[0] if qinfos else {}
                qname = primary.get("question_name")
                qtype = primary.get("question_type")
                # A `select` map filter renders a dropdown of the question's
                # OPTION values (the frontend builds option_equals criteria from
                # them via getQuestionOptions). Only option / multiple_option
                # questions have those. A `geo` question is the map's coordinate
                # source (provided by source_form_id, not a filter) and a
                # number / text question has no options — emitting a select for
                # either produces a broken empty dropdown, so skip them. Their
                # MAP(...) intent (compliance / threshold pin colouring) needs a
                # dedicated filter type and is left for manual authoring.
                if qname and qtype in ("option", "multiple_option"):
                    map_filters.append({
                        "type": "select",
                        "key": qname,
                        "label": r["indicator"].strip(),
                        "form_id": primary["form_id"],
                        "question_name": qname,
                        "color_map": {"__TODO_option__": "#64A73B", "_no_info": "#999999"},
                    })
                elif qname == "num_pumps_standby" and qtype == "number":
                    map_filters.append(_standby_facility_map_filter(
                        primary, r["indicator"].strip()
                    ))
            map_filters.append({
                "type": "toggle",
                "key": "monitored_last_year",
                "label": "Monitored last year",
                "default": True,
                "rolling_months": 12,
            })
            items.append({
                "id": "map_main",
                "chart_type": "map",
                "title": f"Monitored {entity_label.title()} Sites",
                "order": order,
                "col_span": 24,
                "height": 400,
                "source_form_id": form_id,
                "filters": map_filters,
                "click_action": "popup",
                "click_url_template": f"/control-center/data/{form_id}/monitoring/{{data_id}}",
            })
            order += 1

    # At-a-glance sections (top-level, before tabs)
    for g in groups:
        if group_role(g) == "at_a_glance":
            items.append({
                "id": f"title_{_slug(g)}",
                "chart_type": "section_title",
                "order": order,
                "col_span": 24,
                "text": g,
            })
            order += 1
            for r in by_group[g]:
                items.append(build_item(r, order, col_span=12, entity_label=entity_label))
                order += 1

    # Tabs
    TAB_ORDER = [
        "tab_monitoring",
        "tab_water_quality",
        "tab_treatment",
        "tab_individual",
        "tab_all",
        "section",
    ]
    TAB_LABELS = {
        "tab_monitoring":    "Monitoring overview",
        "tab_water_quality": "Water quality",
        "tab_treatment":     "Treatment & Condition",
        "tab_individual":    "Individual Overview",
        "tab_all":           f"All {entity_label.title()} Sites",
        "section":           "Other",
    }

    role_rows = defaultdict(list)
    role_groups = defaultdict(list)
    for g in groups:
        role = group_role(g)
        if role not in ("kpi_cards", "map", "at_a_glance") and by_group[g]:
            role_rows[role].extend(by_group[g])
            role_groups[role].append(g)

    if role_rows:
        tab_items_list = []
        for role in TAB_ORDER:
            if role not in role_rows:
                continue
            tab_inner = []
            tab_order = 1

            if role == "tab_individual":
                component_name = "".join(w.title() for w in entity_label.split())
                tab_inner.append({
                    "id": "individual_overview_component",
                    "chart_type": "custom_component",
                    "order": 1,
                    "component": f"Individual{component_name}Overview",
                })
            elif role == "tab_all":
                tab_inner.append({
                    "id": "table_all",
                    "chart_type": "table",
                    "order": 1,
                    "col_span": 24,
                    "label": f"All {entity_label.title()} Sites",
                    "api": {
                        "form_id": form_id,
                        "monitoring_form_id": "__TODO_monitoring_form_id__",
                        "criteria": [],
                    },
                    "columns": [
                        {"key": "name", "label": "Name",
                         "source": "parent_name", "hide": False},
                        {"key": "administration", "label": "Location",
                         "source": "administration", "hide": False},
                        {"key": "last_monitoring", "label": "Last Monitoring",
                         "source": "latest_date",
                         "question_name": "__TODO_inspection_date_qname__",
                         "hide": False},
                    ],
                })
            else:
                for g in role_groups[role]:
                    if len(role_groups[role]) > 1:
                        tab_inner.append({
                            "id": f"title_{_slug(g)}",
                            "chart_type": "section_title",
                            "order": tab_order,
                            "col_span": 24,
                            "text": g,
                        })
                        tab_order += 1
                    for r in by_group[g]:
                        tab_inner.append(
                            build_item(r, tab_order, col_span=12, entity_label=entity_label)
                        )
                        tab_order += 1

            tab_label = TAB_LABELS[role]
            if role == "tab_treatment" and len(role_groups[role]) == 1:
                tab_label = role_groups[role][0]

            tab_items_list.append({
                "id": f"tab_{_slug(tab_label)}",
                "label": tab_label,
                "items": tab_inner,
            })

        items.append({
            "id": "main_tabs",
            "chart_type": "tabs",
            "order": order,
            "col_span": 24,
            "items": tab_items_list,
        })

    return {
        "parent_form_id": form_id,
        "slug": f"{_slug(form_name)}-overview",
        "name": form_name,
        "description": f"Overview dashboard for {form_name}.",
        "fiscal_year_start_month": 1,
        "items": items,
    }

In [ ]:
# Post-build resolution pass: fill family-derived TODO placeholders
#
# build_config emits skeleton placeholders for references that need the form
# FAMILY (registration + monitoring forms), not a single CSV row:
#   __TODO_list_param_ids__   -> ids of the generated dot_strip param items
#   __TODO_wq_globals_ref__   -> "wq_globals" (a water_quality_globals def we add)
#   __TODO_reg_form_id__      -> the registration (parent) form id
#   __TODO_monitoring_form_id__ -> the comprehensive monitoring form id
#   __TODO_inspection_date_qname__ -> the monitoring date question_name
#   __TODO_option__ in a map filter color_map -> per-option colours from the form
#
# The family is resolved from forms_raw via each monitoring form's `parent_id`
# (the filename pattern <prefix>_<formId>.monitoring.prod.json mirrors this).
# Unresolvable, judgement-call placeholders are intentionally left as TODO:
#   __TODO_unit__ (axis units), __TODO_option_value__ (target_group for % cards),
#   and __TODO_option__ on geo/number map filters (compliance/threshold colouring,
#   which has no option set to derive from).

# Fallback palette for option values that carry no explicit `color` in the form.
_FALLBACK_PALETTE = [
    "#1b9e77", "#d95f02", "#7570b3", "#e7298a",
    "#66a61e", "#e6ab02", "#a6761d", "#666666",
]


def resolve_family(parent_form_id: int):
    """Return (registration_form, comprehensive_monitoring_form, [all_monitoring]).

    Comprehensive = the non-"Quick" monitoring form with the most questions.
    """
    reg = forms_raw.get(parent_form_id)
    monitorings = [
        f for f in forms_raw.values() if f.get("parent_id") == parent_form_id
    ]

    def is_quick(f):
        return "quick" in (f.get("form") or f.get("name", "")).lower()

    def n_questions(f):
        return sum(len(g.get("questions", [])) for g in f.get("question_groups", []))

    pool = [f for f in monitorings if not is_quick(f)] or monitorings
    comprehensive = max(pool, key=n_questions) if pool else None
    return reg, comprehensive, monitorings


def _find_question(form: dict, predicate):
    """First question in `form` matching predicate, else None."""
    if not form:
        return None
    for qg in form.get("question_groups", []):
        for q in qg.get("questions", []):
            if predicate(q):
                return q
    return None


def option_color_map(question_name: str):
    """Build {option_value: colour} for an option/multiple_option question.

    Colours come from each option's `color` property; options without one fall
    back to a stable palette by position. Returns None when the question has no
    options (geo / number / text), so the caller leaves its TODO in place.
    """
    info = qname_map.get(question_name)
    if not info:
        return None
    q = _find_question(forms_raw.get(info["form_id"]),
                       lambda x: x.get("name") == question_name)
    if not q or not q.get("options"):
        return None
    cmap = {}
    for i, opt in enumerate(q["options"]):
        val = opt.get("value")
        if val is None:
            continue
        cmap[val] = opt.get("color") or _FALLBACK_PALETTE[i % len(_FALLBACK_PALETTE)]
    return cmap or None


def _walk(obj, fn):
    """Depth-first visit of every dict node in a nested structure."""
    if isinstance(obj, dict):
        fn(obj)
        for v in obj.values():
            _walk(v, fn)
    elif isinstance(obj, list):
        for it in obj:
            _walk(it, fn)


def resolve_todos(config: dict) -> dict:
    """Fill family-derived placeholders in a generated config (in place)."""
    parent_id = config["parent_form_id"]
    reg, comp, _monitorings = resolve_family(parent_id)
    comp_id = comp["id"] if comp else None

    # Monitoring date question (first date-type in the comprehensive monitoring form).
    date_q = _find_question(comp, lambda q: q.get("type") == "date")
    date_qname = date_q["name"] if date_q else None

    # Water-quality globals: sample + test-method questions. Names vary across
    # families (can_take_water_sample / can_take_sample / can_collect_water_sample;
    # water_testing_method / effluent_test_method), so match on a stable substring.
    sample_q = _find_question(
        comp, lambda q: q.get("type") == "option" and q.get("name", "").endswith("water_sample")
    ) or _find_question(
        comp, lambda q: q.get("type") == "option" and "sample" in q.get("name", "")
    )
    method_q = _find_question(
        comp, lambda q: q.get("name") == "water_testing_method"
    ) or _find_question(
        comp,
        lambda q: q.get("type") in ("option", "multiple_option")
        and ("test_method" in q.get("name", "") or "testing_method" in q.get("name", "")),
    )

    # Collect ids of generated water-quality dot_strip param items.
    param_ids = []
    _walk(config, lambda d: param_ids.append(d["id"])
          if d.get("chart_type") == "dot_strip" and isinstance(d.get("id"), str) else None)

    # Add a water_quality_globals definition (only if there are params to comply with).
    items = config["items"]
    has_globals = any(
        isinstance(it, dict) and it.get("chart_type") == "water_quality_globals"
        for it in items
    )
    if param_ids and not has_globals and (sample_q or method_q):
        items.insert(0, {
            "id": "wq_globals",
            "chart_type": "water_quality_globals",
            "hide": True,
            "order": 0,
            "sample_question_name": sample_q["name"] if sample_q else "__TODO_sample_qname__",
            "test_method_question_name": method_q["name"] if method_q else "__TODO_test_method_qname__",
            "monitoring_form_id": comp_id or "__TODO_monitoring_form_id__",
        })

    # Replace scalar placeholder tokens throughout the tree.
    def fill(node: dict):
        for k, v in list(node.items()):
            if v == ["__TODO_list_param_ids__"] and param_ids:
                node[k] = list(param_ids)
            elif v == "__TODO_wq_globals_ref__":
                node[k] = "wq_globals"
            elif v == "__TODO_reg_form_id__":
                node[k] = parent_id
            elif v == "__TODO_monitoring_form_id__" and comp_id:
                node[k] = comp_id
            elif v == "__TODO_inspection_date_qname__" and date_qname:
                node[k] = date_qname

    _walk(config, fill)

    # Resolve map-filter colour maps from option colours. Placeholder color_map is
    # {"__TODO_option__": ..., "_no_info": ...}; for option/multiple_option questions
    # replace it with one entry per option value. Non-option questions keep the TODO.
    def fill_colors(node: dict):
        cmap = node.get("color_map")
        if isinstance(cmap, dict) and "__TODO_option__" in cmap and node.get("question_name"):
            opts = option_color_map(node["question_name"])
            if opts:
                node["color_map"] = {**opts, "_no_info": cmap.get("_no_info", "#999999")}

    _walk(config, fill_colors)
    return config


# Quick check against the WAF family (parent 1749634736797).
_reg, _comp, _mons = resolve_family(1749634736797)
if _comp:
    print(f"WAF comprehensive monitoring: {_comp['id']}  {_comp.get('form')}")
    print(f"  monitoring forms in family : {[m['id'] for m in _mons]}")
    print(f"  disinfection_technique colours: {option_color_map('disinfection_technique')}")
    print(f"  final_recommendations colours : {option_color_map('final_recommendations')}")

In [ ]:
# Generate and write all configs

ENTITY_LABELS = {
    "1748903240763": "WWTP",
    "1749611049520": "Pump Station",
    "1749634736797": "WTP",
}

for csv_file in sorted(SOURCE_DIR.glob("*.csv")):
    form_id = int(csv_file.stem)
    entity_label = ENTITY_LABELS.get(str(form_id), "site")

    form_name = fid_map.get(form_id, f"Form {form_id}")

    rows = load_rows(csv_file)

    by_group = defaultdict(list)
    for r in rows:
        by_group[r["group"]].append(r)

    print(f"\n{'='*60}")
    print(f"Form {form_id}: {form_name}  ({entity_label})")
    print(f"Rows: {len(rows)}")
    for g, rs in by_group.items():
        print(f"  [{group_role(g):20}] {g!r:45} ({len(rs)} rows)")

    config = build_config(form_id, rows, form_name, entity_label)
    config = resolve_todos(config)  # fill family-derived placeholders

    out_path = OUTPUT_DIR / f"{form_id}.json"
    out_path.write_text(json.dumps(config, indent=2, ensure_ascii=False))
    print(f"\nWritten -> {out_path}  ({len(config['items'])} top-level items)")

In [ ]:
# Summary: count items using question_name vs form_id+question_id

def count_api_modes(obj, counts=None):
    if counts is None:
        counts = {"question_name": 0, "form_id_qid": 0, "form_id_only": 0, "other": 0}
    if isinstance(obj, dict):
        api = obj.get("api", {})
        ct = obj.get("chart_type", "")
        skip = {"filter_bar", "section_title", "tabs", "filter_date",
                "filter_administration", "filter_option", "filter_multi_option", ""}
        if ct not in skip and isinstance(api, dict):
            if "question_name" in api:
                counts["question_name"] += 1
            elif "question_id" in api:
                counts["form_id_qid"] += 1
            elif "form_id" in api:
                counts["form_id_only"] += 1
            elif ct not in ("custom_component", "__TODO__"):
                counts["other"] += 1
        for v in obj.values():
            if isinstance(v, (dict, list)):
                count_api_modes(v, counts)
    elif isinstance(obj, list):
        for item in obj:
            count_api_modes(item, counts)
    return counts


print(f"\n{'API mode summary':=^55}")
for csv_file in sorted(SOURCE_DIR.glob("*.csv")):
    out_path = OUTPUT_DIR / f"{csv_file.stem}.json"
    if not out_path.exists():
        continue
    config = json.loads(out_path.read_text())
    counts = count_api_modes(config)
    total = sum(counts.values())
    print(f"\n{csv_file.stem}  (total items with api: {total})")
    for mode, n in counts.items():
        pct = f"{100*n/total:.0f}%" if total else ""
        print(f"  {mode:20}: {n:3}  {pct}")